# Evaluation Metrics

## Learning Objectives
1. Derive precision, recall, F1, and accuracy from a confusion matrix using only NumPy.
2. Use sklearn's full metrics suite including ROC-AUC, PR-AUC, and multi-class reports.
3. Understand when macro, micro, and weighted averaging matter for imbalanced datasets.
4. Align model metrics with business objectives by plotting precision-recall tradeoff curves.


In [ ]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import torch

np.random.seed(42)
torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression


## Level 1: Classification Metrics from Scratch (NumPy)

In [ ]:
def confusion_matrix_np(y_true, y_pred, n_classes=None):
    """Build confusion matrix without sklearn."""
    if n_classes is None:
        n_classes = max(y_true.max(), y_pred.max()) + 1
    cm = np.zeros((n_classes, n_classes), dtype=int)
    for t, p in zip(y_true, y_pred):
        cm[t, p] += 1
    return cm


def metrics_from_cm(cm):
    """Compute per-class precision, recall, F1 and overall accuracy."""
    n = cm.shape[0]
    tp = np.diag(cm)
    fp = cm.sum(axis=0) - tp   # column sums minus diagonal
    fn = cm.sum(axis=1) - tp   # row sums minus diagonal

    precision = np.where(tp + fp > 0, tp / (tp + fp), 0.0)
    recall = np.where(tp + fn > 0, tp / (tp + fn), 0.0)
    f1 = np.where(precision + recall > 0,
                  2 * precision * recall / (precision + recall), 0.0)
    accuracy = tp.sum() / cm.sum()
    return precision, recall, f1, accuracy


# Synthetic binary data
y_true = np.array([0]*60 + [1]*40)
np.random.shuffle(y_true)
noise_idx = np.random.choice(len(y_true), size=15, replace=False)
y_pred = y_true.copy()
y_pred[noise_idx] = 1 - y_pred[noise_idx]

cm = confusion_matrix_np(y_true, y_pred)
prec, rec, f1_score, acc = metrics_from_cm(cm)

print("Confusion matrix:
", cm)
print(f"
Class 0 — Precision: {prec[0]:.3f}  Recall: {rec[0]:.3f}  F1: {f1_score[0]:.3f}")
print(f"Class 1 — Precision: {prec[1]:.3f}  Recall: {rec[1]:.3f}  F1: {f1_score[1]:.3f}")
print(f"Accuracy: {acc:.3f}")
print(f"Macro F1: {f1_score.mean():.3f}")


## Level 2: Full Metrics Suite with sklearn

In [ ]:
from sklearn.metrics import (
    classification_report, roc_auc_score, average_precision_score,
    confusion_matrix, roc_curve, precision_recall_curve,
)

# Generate a moderately imbalanced binary dataset
X, y = make_classification(
    n_samples=1000, n_features=15, n_informative=8,
    weights=[0.7, 0.3], random_state=42
)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

lr = LogisticRegression(max_iter=500, random_state=42)
lr.fit(X_train, y_train)
y_pred = lr.predict(X_test)
y_prob = lr.predict_proba(X_test)[:, 1]

print("=" * 55)
print("Classification Report:")
print(classification_report(y_test, y_pred, target_names=["negative", "positive"]))

roc_auc = roc_auc_score(y_test, y_prob)
pr_auc = average_precision_score(y_test, y_prob)
print(f"ROC-AUC : {roc_auc:.4f}  (1.0 = perfect, 0.5 = random)")
print(f"PR-AUC  : {pr_auc:.4f}  (equals prevalence for random classifier)")

cm = confusion_matrix(y_test, y_pred)
print("
Confusion matrix (rows=true, cols=pred):")
print(cm)
print(f"
True Positives: {cm[1,1]}  False Negatives: {cm[1,0]}")
print(f"False Positives: {cm[0,1]}  True Negatives: {cm[0,0]}")


## Real-World Example 1: Macro vs Micro vs Weighted F1 (Imbalanced Classes)

In [ ]:
# Macro: average each class equally (penalises poor minority-class performance)
# Micro: aggregate TP/FP/FN globally (dominated by majority class)
# Weighted: weight each class by its support (balanced between macro and micro)

from sklearn.metrics import f1_score

# Build a 3-class imbalanced dataset
X3, y3 = make_classification(
    n_samples=1500, n_features=12, n_informative=6,
    n_classes=3, n_clusters_per_class=1,
    weights=[0.6, 0.3, 0.1],   # 60/30/10 split
    random_state=42
)
X3_tr, X3_te, y3_tr, y3_te = train_test_split(X3, y3, test_size=0.2, random_state=42)

lr3 = LogisticRegression(max_iter=500, random_state=42)
lr3.fit(X3_tr, y3_tr)
y3_pred = lr3.predict(X3_te)

for avg in ["macro", "micro", "weighted"]:
    score = f1_score(y3_te, y3_pred, average=avg)
    print(f"F1 ({avg:8s}): {score:.4f}")

print()
print("Per-class support:")
for cls in range(3):
    support = (y3_te == cls).sum()
    cls_f1 = f1_score(y3_te == cls, y3_pred == cls)
    print(f"  Class {cls}: support={support:3d}  F1={cls_f1:.4f}")

print("
Key insight:")
print("  Macro treats rare class equally — use when minority errors are costly.")
print("  Weighted hides minority failure — use for overall system health monitoring.")


## Real-World Example 2: Regression Metrics

In [ ]:
# Common regression metrics: MSE, MAE, RMSE, R², MAPE
# Huber loss bridges MSE (sensitive to outliers) and MAE (robust but non-smooth).

from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

def mean_absolute_percentage_error(y_true, y_pred):
    """MAPE — interpretable as % error. Fails when y_true has zeros."""
    mask = y_true != 0
    return np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100


def huber_loss(y_true, y_pred, delta=1.0):
    """Huber: quadratic for |e|<=delta, linear beyond."""
    e = np.abs(y_true - y_pred)
    return np.where(e <= delta, 0.5 * e**2, delta * (e - 0.5 * delta)).mean()


# Synthetic regression targets with a few outliers
n = 200
y_true_reg = 3.0 * np.random.randn(n) + 10.0
y_pred_reg = y_true_reg + np.random.normal(0, 1.0, n)
# Inject 5 outliers
outlier_idx = np.random.choice(n, 5, replace=False)
y_pred_reg[outlier_idx] += np.random.choice([-10, 10], 5)

mse = mean_squared_error(y_true_reg, y_pred_reg)
mae = mean_absolute_error(y_true_reg, y_pred_reg)
rmse = np.sqrt(mse)
r2 = r2_score(y_true_reg, y_pred_reg)
mape = mean_absolute_percentage_error(y_true_reg, y_pred_reg)
huber = huber_loss(y_true_reg, y_pred_reg, delta=2.0)

print(f"MSE  : {mse:.4f}   (penalises outliers quadratically)")
print(f"RMSE : {rmse:.4f}  (same units as target, still outlier-sensitive)")
print(f"MAE  : {mae:.4f}   (robust to outliers, linear penalty)")
print(f"Huber: {huber:.4f}  (blend: robust for large errors, smooth near zero)")
print(f"R²   : {r2:.4f}    (explained variance; 1.0 = perfect)")
print(f"MAPE : {mape:.2f}%  (interpretable %, fails at zero targets)")


## Real-World Example 3: Precision-Recall Tradeoff for Business Objectives

In [ ]:
# Different tasks require different operating points on the PR curve.
# Spam filter → maximise precision (low false-positive rate, user trust)
# Cancer screening → maximise recall (low false-negative rate, patient safety)

from sklearn.metrics import precision_recall_curve

precisions, recalls, thresholds = precision_recall_curve(y_test, y_prob)

# Find threshold maximising F1
f1_values = 2 * precisions * recalls / np.where(precisions + recalls > 0, precisions + recalls, 1)
best_f1_idx = np.argmax(f1_values[:-1])  # exclude last point (precision=1, recall=0)

# High-precision operating point (spam filter: precision >= 0.95)
high_prec_idx = np.searchsorted(-precisions, -0.95)
# High-recall operating point (screening: recall >= 0.90)
high_rec_idx = np.searchsorted(-recalls[::-1], -0.90)
high_rec_idx = len(recalls) - 1 - high_rec_idx

print("Operating point comparison:")
print(f"{'Objective':<25} {'Threshold':>10} {'Precision':>10} {'Recall':>8} {'F1':>8}")
print("-" * 65)

for label, idx in [("Max F1", best_f1_idx),
                   ("Spam filter (P≥0.95)", min(high_prec_idx, len(thresholds)-1)),
                   ("Screening (R≥0.90)", min(high_rec_idx, len(thresholds)-1))]:
    thr = thresholds[idx] if idx < len(thresholds) else thresholds[-1]
    p = precisions[idx]
    r = recalls[idx]
    f = 2*p*r/(p+r) if p+r > 0 else 0
    print(f"{label:<25} {thr:>10.3f} {p:>10.3f} {r:>8.3f} {f:>8.3f}")

print(f"
AU-PRC: {average_precision_score(y_test, y_prob):.4f}")
print("Lesson: choose threshold based on cost asymmetry, not just max-F1.")
